In [1]:
from ingest import load_faq_data

documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)
        

153

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [9]:
import json

user_prompt = json.dumps(doc)

In [12]:
print(type(user_prompt))
print(user_prompt)

<class 'str'>
{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."}


In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
response.output_parsed.questions

['I just found this course — can I still jump in now, or is it too late?',
 'Is it okay to join the course after it has already started?',
 'If I start the course late, can I still get a certificate somehow?',
 'Do I need to submit the project before submissions close in order to get the certificate?',
 'Can I still earn a certificate if I only join the course now, or did I miss the deadline?']

In [16]:
len(documents)

153

In [17]:
from evaluation_utils import llm_structured
from evaluation_utils import calc_price

In [18]:
result, answer_usage= llm_structured(
    openai_client, 
    data_gen_instructions,
    user_prompt,
    Questions
)

In [20]:
answer_usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=289)

In [21]:
calc_price(answer_usage)

{'input_cost': 0.00015525, 'output_cost': 0.000369, 'total_cost': 0.00052425}

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit the project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for the project if I want the course certificate?',
  'document': '74eb249bbf'}]

In [25]:
import pandas as pd

In [26]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Is it too late to start this course now?,74eb249bbf
2,"If I join after the course started, can I stil...",74eb249bbf
3,Do I need to submit the project before submiss...,74eb249bbf
4,What’s the deadline for the project if I want ...,74eb249bbf
